
# 03 — Consolidação dos Resultados — UTKFace

Este notebook tem como objetivo consolidar e analisar os resultados obtidos pelas tecnologias DeepFace e MiVOLO durante o processamento da base de imagens UTKFace.

São utilizados os arquivos CSV gerados pelos notebooks de inferência, contemplando as três partições da base de dados (`part1`, `part2` e `part3`) para cada tecnologia.

A partir desses arquivos, são realizados os seguintes procedimentos:

- Carregamento e validação dos arquivos de resultados;
- Verificação da quantidade de registros e da ocorrência de duplicidades;
- Comparação dos arquivos processados pelas duas tecnologias;
- Consolidação dos resultados por partição e por tecnologia;
- Cálculo do Erro Absoluto Médio (MAE) por partição e global;
- Análise da quantidade e da cobertura das estimativas numéricas;
- Identificação das estimativas negativas e dos registros sem decisão válida;
- Exportação das tabelas consolidadas para arquivos CSV.

O cálculo do MAE considera todas as estimativas numéricas finitas, inclusive aquelas com valores negativos. Entretanto, estimativas negativas não são consideradas decisões válidas nas análises de cobertura de decisões.

Este notebook tem finalidade de consolidação e diagnóstico dos resultados experimentais. As análises de classificação por faixas etárias, limiares e matrizes de confusão são desenvolvidas separadamente no Notebook 05.


In [ ]:

from google.colab import drive

drive.mount("/content/drive")


In [ ]:

## 1. Bibliotecas, caminhos e parâmetros

import os
import numpy as np
import pandas as pd

# ============================================================
# CAMINHOS DOS RESULTADOS
# ============================================================

PASTA_DEEPFACE = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/"
    "resultados/deepface"
)

PASTA_MIVOLO = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/resultados/MiVOLO"
)

PASTA_SAIDA = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/"
    "resultados"
)

os.makedirs(PASTA_SAIDA, exist_ok=True)

PARTES = ["part1", "part2", "part3"]

COLUNAS_OBRIGATORIAS = [
    "Arquivo",
    "Idade Real",
    "Idade Estimada",
    "Erro Absoluto",
    "Status"
]

ARQUIVOS = {
    "DeepFace": {
        parte: os.path.join(
            PASTA_DEEPFACE,
            f"resultados_deepface_utkface_{parte}.csv"
        )
        for parte in PARTES
    },
    "MiVOLO": {
        parte: os.path.join(
            PASTA_MIVOLO,
            f"resultados_mivolo_utkface_{parte}.csv"
        )
        for parte in PARTES
    }
}

STATUS_SUCESSO = {
    "DeepFace": "Sucesso",
    "MiVOLO": "OK"
}

print("Configuração concluída.")
print("Diretório de saída:", PASTA_SAIDA)


In [ ]:

## 2. Carregamento, validação e tratamento dos registros

dados_por_tecnologia = {}
diagnostico_arquivos = []

for tecnologia, arquivos_parte in ARQUIVOS.items():

    partes_carregadas = []

    for parte, caminho in arquivos_parte.items():

        if not os.path.isfile(caminho):
            raise FileNotFoundError(
                f"Arquivo não encontrado: {caminho}"
            )

        df = pd.read_csv(caminho)

        faltantes = [
            coluna for coluna in COLUNAS_OBRIGATORIAS
            if coluna not in df.columns
        ]

        if faltantes:
            raise ValueError(
                f"Colunas ausentes em {caminho}: {faltantes}"
            )

        df = df[COLUNAS_OBRIGATORIAS].copy()

        df["Tecnologia"] = tecnologia
        df["Parte"] = parte

        # Padronização dos campos utilizados.
        df["Arquivo"] = (
            df["Arquivo"].astype("string").str.strip()
        )

        for coluna in [
            "Idade Real",
            "Idade Estimada",
            "Erro Absoluto"
        ]:
            df[coluna] = pd.to_numeric(
                df[coluna],
                errors="coerce"
            )

        # Identificação das inferências bem-sucedidas.
        status_esperado = STATUS_SUCESSO[tecnologia]

        df["Status_Sucesso"] = (
            df["Status"].astype(str).str.strip()
            == status_esperado
        )

        # Saída numérica: status de sucesso e idades finitas.
        df["Saida_Numerica"] = (
            df["Status_Sucesso"]
            & np.isfinite(df["Idade Real"])
            & np.isfinite(df["Idade Estimada"])
        )

        # Recalcula o erro a partir das idades disponíveis.
        df["Erro_Com_Sinal"] = np.where(
            df["Saida_Numerica"],
            df["Idade Estimada"] - df["Idade Real"],
            np.nan
        )

        df["Erro_Absoluto_Recalculado"] = (
            df["Erro_Com_Sinal"].abs()
        )

        # Estimativas negativas não constituem decisões válidas.
        df["Decisao_Valida"] = (
            df["Saida_Numerica"]
            & (df["Idade Estimada"] >= 0)
        )

        partes_carregadas.append(df)

        diagnostico_arquivos.append({
            "Tecnologia": tecnologia,
            "Parte": parte,
            "Registros": len(df),
            "Nomes_distintos": df["Arquivo"].nunique(
                dropna=True
            ),
            "Nomes_ausentes": int(
                df["Arquivo"].isna().sum()
            ),
            "Duplicidades_de_arquivo": int(
                df["Arquivo"].duplicated().sum()
            ),
            "Status_sucesso": int(
                df["Status_Sucesso"].sum()
            ),
            "Saidas_numericas": int(
                df["Saida_Numerica"].sum()
            ),
            "Estimativas_negativas": int(
                (
                    df["Saida_Numerica"]
                    & (df["Idade Estimada"] < 0)
                ).sum()
            ),
            "Decisoes_validas": int(
                df["Decisao_Valida"].sum()
            )
        })

    dados_por_tecnologia[tecnologia] = pd.concat(
        partes_carregadas,
        ignore_index=True
    )

df_diagnostico_arquivos = pd.DataFrame(
    diagnostico_arquivos
)

display(df_diagnostico_arquivos)



In [ ]:

## 3. Verificação de duplicidades entre partições

duplicidades_globais = []

for tecnologia, df_tec in dados_por_tecnologia.items():

    repetidos = df_tec[
        df_tec["Arquivo"].duplicated(keep=False)
    ].copy()

    duplicidades_globais.append({
        "Tecnologia": tecnologia,
        "Total_registros": len(df_tec),
        "Nomes_distintos": df_tec["Arquivo"].nunique(
            dropna=True
        ),
        "Registros_com_nome_repetido": len(repetidos),
        "Nomes_repetidos_distintos": (
            repetidos["Arquivo"].nunique(dropna=True)
        )
    })

df_duplicidades_globais = pd.DataFrame(
    duplicidades_globais
)

display(df_duplicidades_globais)


In [ ]:

## 4. Verificação da correspondência entre os resultados

A comparação é realizada pelos nomes dos arquivos. A correspondência indica que as duas tecnologias possuem registros associados aos mesmos nomes de imagem, mas não comprova ausência de duplicatas visuais ou identidade das pessoas retratadas.

def conjunto_nomes(df):
    return set(
        df["Arquivo"].dropna().astype(str)
    )


nomes_deepface = conjunto_nomes(
    dados_por_tecnologia["DeepFace"]
)

nomes_mivolo = conjunto_nomes(
    dados_por_tecnologia["MiVOLO"]
)

somente_deepface = sorted(
    nomes_deepface - nomes_mivolo
)

somente_mivolo = sorted(
    nomes_mivolo - nomes_deepface
)

nomes_comuns = nomes_deepface & nomes_mivolo

comparacao = pd.DataFrame([{
    "Nomes distintos DeepFace": len(nomes_deepface),
    "Nomes distintos MiVOLO": len(nomes_mivolo),
    "Nomes em comum": len(nomes_comuns),
    "Somente DeepFace": len(somente_deepface),
    "Somente MiVOLO": len(somente_mivolo)
}])

display(comparacao)

if somente_deepface:
    print(
        "Exemplos presentes somente no DeepFace:",
        somente_deepface[:10]
    )

if somente_mivolo:
    print(
        "Exemplos presentes somente no MiVOLO:",
        somente_mivolo[:10]
    )


In [ ]:

## 5. Consolidação por partição e por tecnologia

O MAE é calculado pela média dos erros absolutos individuais. O resultado global é calculado sobre todos os registros numéricos das três partições reunidas, não correspondendo à média simples dos três MAEs de partição.

As coberturas utilizam como denominador o total de registros da respectiva partição ou tecnologia. Estimativas negativas são consideradas saídas numéricas para o cálculo do MAE, mas não são contabilizadas como decisões válidas.

resultados_mae = []
resultados_cobertura = []

for tecnologia, df_tec in dados_por_tecnologia.items():

    grupos = [
        (
            parte,
            df_tec[df_tec["Parte"] == parte].copy()
        )
        for parte in PARTES
    ]

    grupos.append(("Consolidado", df_tec.copy()))

    for conjunto, grupo in grupos:

        total = len(grupo)

        numericas = grupo[
            grupo["Saida_Numerica"]
        ]

        validas = grupo[
            grupo["Decisao_Valida"]
        ]

        mae = numericas[
            "Erro_Absoluto_Recalculado"
        ].mean()

        resultados_mae.append({
            "Tecnologia": tecnologia,
            "Base": "UTKFace",
            "Conjunto": conjunto,
            "Total_registros": total,
            "Quantidade_saidas_numericas": len(numericas),
            "MAE_anos": mae
        })

        resultados_cobertura.append({
            "Tecnologia": tecnologia,
            "Base": "UTKFace",
            "Conjunto": conjunto,
            "Total_registros": total,
            "Saidas_numericas": len(numericas),
            "Estimativas_negativas": int(
                (
                    numericas["Idade Estimada"] < 0
                ).sum()
            ),
            "Decisoes_validas": len(validas),
            "Sem_decisao_valida": total - len(validas),
            "Cobertura_numerica_pct": (
                100 * len(numericas) / total
                if total else np.nan
            ),
            "Cobertura_decisoes_validas_pct": (
                100 * len(validas) / total
                if total else np.nan
            )
        })


df_mae = pd.DataFrame(resultados_mae)

df_cobertura = pd.DataFrame(
    resultados_cobertura
)

df_mae["MAE_anos"] = (
    df_mae["MAE_anos"].round(2)
)

df_cobertura[
    [
        "Cobertura_numerica_pct",
        "Cobertura_decisoes_validas_pct"
    ]
] = df_cobertura[
    [
        "Cobertura_numerica_pct",
        "Cobertura_decisoes_validas_pct"
    ]
].round(2)


print("MAE por partição e consolidado:")
display(df_mae)

print("Cobertura numérica e de decisões válidas:")
display(df_cobertura)


In [ ]:

## 6. Verificação da consistência do erro absoluto exportado

Os notebooks de inferência exportam o erro absoluto arredondado a duas casas decimais. Esta verificação compara o valor registrado no CSV com o erro absoluto recalculado a partir das idades disponíveis. Pequenas diferenças de precisão podem decorrer do arredondamento realizado durante a exportação.

verificacoes_erro = []

for tecnologia, df_tec in dados_por_tecnologia.items():

    validos = df_tec[
        df_tec["Saida_Numerica"]
    ].copy()

    diferenca = (
        validos["Erro Absoluto"]
        - validos["Erro_Absoluto_Recalculado"]
    ).abs()

    verificacoes_erro.append({
        "Tecnologia": tecnologia,
        "Saidas_numericas": len(validos),
        "Erro_absoluto_ausente_no_csv": int(
            validos["Erro Absoluto"].isna().sum()
        ),
        "Diferenca_maxima_anos": diferenca.max(),
        "Registros_com_diferenca_maior_que_0_01": int(
            (diferenca > 0.01).sum()
        )
    })

df_verificacao_erro = pd.DataFrame(
    verificacoes_erro
)

display(df_verificacao_erro)


In [ ]:

## 7. Exportação das tabelas consolidadas

exportacoes = {
    "resultados_mae_utkface.csv": df_mae,
    "resultados_cobertura_utkface.csv": df_cobertura,
    "diagnostico_arquivos_utkface.csv": df_diagnostico_arquivos,
    "comparacao_nomes_tecnologias_utkface.csv": comparacao,
    "verificacao_erros_utkface.csv": df_verificacao_erro,
    "duplicidades_globais_utkface.csv": df_duplicidades_globais
}

for nome, tabela in exportacoes.items():

    destino = os.path.join(
        PASTA_SAIDA,
        nome
    )

    tabela.to_csv(
        destino,
        index=False,
        encoding="utf-8-sig"
    )

    print("Arquivo salvo:", destino)

print("\nConsolidação concluída.")


In [ ]:

## 8. Critérios para interpretação

- O MAE inclui todas as saídas numéricas finitas de registros com status de sucesso, inclusive estimativas negativas.
- A cobertura numérica representa a proporção de registros com saída numérica finita.
- A cobertura de decisões válidas exige, adicionalmente, que a idade estimada seja não negativa.
- Registros sem decisão válida permanecem no denominador das coberturas.
- O MAE consolidado é calculado diretamente sobre os registros das três partições reunidas, considerando a quantidade efetiva de saídas numéricas de cada partição.
- A correspondência entre nomes de arquivos não permite concluir que não existam duplicatas visuais ou fotografias da mesma pessoa.
